# Лабораторная работа №4. Использование библиотеки pandas
## Задание 3. Чемпионы «Формулы-1»
Источник: [Wikipedia — World Drivers’ Champions, таблица By season](https://en.wikipedia.org/wiki/List_of_Formula_One_World_Drivers%27_Champions).
Снимок таблицы загружен 10.10.2026 и сохранён в `formula1_source.html` рядом с блокнотом.
По умолчанию используем снимок для воспроизводимости; `REFRESH_FROM_WEB = True` загружает страницу заново.
Необходимы Python, pandas и lxml.

## 1. Загрузка таблицы чемпионов по сезонам

In [1]:
import pandas as pd
from pathlib import Path
from io import StringIO
from urllib.request import Request, urlopen
from IPython.display import display

URL = "https://en.wikipedia.org/wiki/List_of_Formula_One_World_Drivers%27_Champions"
REFRESH_FROM_WEB = False
cache = Path('formula1_source.html')
if REFRESH_FROM_WEB or not cache.exists():
    request = Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urlopen(request, timeout=30) as response:
        html = response.read().decode('utf-8')
    tables = pd.read_html(StringIO(html))
    raw = next(t for t in tables if any('Clinched' in str(c) for c in t.columns))
    cache.write_text(raw.to_html(index=False), encoding='utf-8')
else:
    raw = pd.read_html(StringIO(cache.read_text(encoding='utf-8')), header=[0, 1])[0]
print('Размер исходной таблицы:', raw.shape)
display(raw.head())
display(raw.tail())

Размер исходной таблицы: (80, 16)


Season                  Driver Age Constructor             Tyres Poles  \
    Season                  Driver Age     Chassis      Engine Tyres Poles   
0     1950     Giuseppe Farina[23]  44  Alfa Romeo  Alfa Romeo     P     2   
1     1951  Juan Manuel Fangio[24]  40  Alfa Romeo  Alfa Romeo     P     4   
2  1952[a]      Alberto Ascari[26]  34     Ferrari     Ferrari   F P     5   
3  1953[a]      Alberto Ascari[26]  35     Ferrari     Ferrari     P     6   
4     1954  Juan Manuel Fangio[24]  43    Maserati    Maserati     P     5   

  Wins Podiums Fastest laps Points          % Points  Clinched[17]  \
  Wins Podiums Fastest laps Points          % Points  Clinched[17]   
0    3       3            3     30   83.333 (47.619)  Round 7 of 7   
1    3       5            5     31   86.111 (51.389)  Round 8 of 8   
2    6       6            6     36  100.000 (74.306)  Round 6 of 8   
3    5       5            4   34.5   95.833 (57.407)  Round 8 of 9   
4    6       7            3     42   93.333 (70.547)  Round 7 of 9   

  # of rounds remaining  Margin % Margin  
  # of rounds remaining  Margin % Margin  
0                     0       3   10.000  
1                     0       6   19.355  
2                     2      12   33.333  
3                     1     6.5   18.841  
4                     2  16.857   40.136

Season              Driver  Age  Constructor               Tyres  Poles  \
    Season              Driver  Age      Chassis       Engine  Tyres  Poles   
75    2024  Max Verstappen[60]   27     Red Bull   Honda RBPT      P      8   
76     NaN                 NaN  NaN          NaN          NaN    NaN    NaN   
77    2025    Lando Norris[61]   26      McLaren     Mercedes      P      7   
78  Season              Driver  Age      Chassis       Engine  Tyres  Poles   
79  Season              Driver  Age  Constructor  Constructor  Tyres  Poles   

    Wins  Podiums  Fastest laps  Points  % Points    Clinched[17]  \
    Wins  Podiums  Fastest laps  Points  % Points    Clinched[17]   
75     9       14             3     437    65.030  Round 22 of 24   
76   NaN      NaN           NaN     NaN       NaN             NaN   
77     7       18             6     423    65.278  Round 24 of 24   
78  Wins  Podiums  Fastest laps  Points  % Points        Clinched   
79  Wins  Podiums  Fastest laps  Points  % Points        Clinched   

    # of rounds remaining  Margin  % Margin  
    # of rounds remaining  Margin  % Margin  
75                      2      63    14.416  
76                    NaN     NaN       NaN  
77                      0       2     0.473  
78  # of rounds remaining  Margin  % Margin  
79  # of rounds remaining  Margin  % Margin

## 2. Удаление служебных строк и сносок
Выбираем строки, у которых сезон — четырёхзначный год. Это удаляет повтор заголовков и пустые разделители независимо от их положения.
Исходная таблица имеет два уровня заголовков: для конструктора сохраняем отдельно **Chassis** и **Engine**.

In [2]:
data = raw.copy()
data.columns = [str(c[-1]) if isinstance(c, tuple) else str(c) for c in data.columns]
data.columns = data.columns.str.replace(r'\[[^\]]*\]', '', regex=True).str.strip()
for column in data.columns:
    data[column] = data[column].astype('string').str.replace(r'\[[^\]]*\]', '', regex=True).str.strip()
data = data.loc[data['Season'].str.fullmatch(r'\d{4}', na=False)].copy()
print('Удалено служебных строк:', len(raw) - len(data))
display(data.tail())

Удалено служебных строк: 3


,Season,Driver,Age,Chassis,Engine,Tyres,Poles,Wins,Podiums,Fastest laps,Points,% Points,Clinched,# of rounds remaining,Margin,% Margin
72,2021,Max Verstappen,24,Red Bull,Honda,P,10,10,18,6,395.5,69.692,Round 22 of 22,0,8,2.023
73,2022,Max Verstappen,25,Red Bull,RBPT,P,7,15,17,5,454,76.174,Round 18 of 22,4,146,32.159
74,2023,Max Verstappen,26,Red Bull,Honda RBPT,P,12,19,21,9,575,92.742,Round 17 of 22,5,290,50.435
75,2024,Max Verstappen,27,Red Bull,Honda RBPT,P,8,9,14,3,437,65.030,Round 22 of 24,2,63,14.416
77,2025,Lando Norris,26,McLaren,Mercedes,P,7,7,18,6,423,65.278,Round 24 of 24,0,2,0.473


## 3. Преобразование числовых столбцов
Из `Clinched` извлекаем номер гонки, в которой завоёван титул, и общее количество гонок сезона.
В полях очков и процентов бывают значения в скобках: основное значение и дополнительное сохраняем отдельно.

In [3]:
display(data.dtypes.to_frame('Тип до преобразования'))
numeric = ['Season', 'Age', 'Poles', 'Wins', 'Podiums', 'Fastest laps', 'Points', '% Points', '# of rounds remaining', 'Margin', '% Margin']
for column in numeric:
    text = data[column].str.replace(',', '', regex=False)
    if text.str.contains(r'\(', na=False).any():
        data[column + ' (в скобках)'] = pd.to_numeric(text.str.extract(r'\(([0-9.]+)\)', expand=False), errors='coerce')
    data[column] = pd.to_numeric(text.str.extract(r'([-+]?\d+(?:\.\d+)?)', expand=False), errors='raise')
rounds = data['Clinched'].str.extract(r'Round\s+(\d+)\s+of\s+(\d+)')
data['Title round'] = pd.to_numeric(rounds[0], errors='raise')
data['Races'] = pd.to_numeric(rounds[1], errors='raise')
assert data[numeric + ['Races']].notna().all().all(), 'Обнаружены нераспознанные числовые значения'
data['Season'] = data['Season'].astype(int)
# В 1954 году Фанхио выступал за двух конструкторов: это один личный титул.
seasons = data.drop_duplicates('Season').sort_values('Season').copy()
display(data.dtypes.to_frame('Тип после преобразования'))
print(f"Период: {seasons['Season'].min()}–{seasons['Season'].max()}; сезонов: {len(seasons)}.")
display(seasons.head())

,Тип до преобразования
Season,string
Driver,string
Age,string
Chassis,string
Engine,string
Tyres,string
Poles,string
Wins,string
Podiums,string
Fastest laps,string


,Тип после преобразования
Season,int64
Driver,string
Age,Int64
Chassis,string
Engine,string
Tyres,string
Poles,Int64
Wins,Int64
Podiums,Int64
Fastest laps,Int64


Период: 1950–2025; сезонов: 76.


,Season,Driver,Age,Chassis,Engine,Tyres,Poles,Wins,Podiums,Fastest laps,Points,% Points,Clinched,# of rounds remaining,Margin,% Margin,% Points (в скобках),Title round,Races
0,1950,Giuseppe Farina,44,Alfa Romeo,Alfa Romeo,P,2,3,3,3,30.0,83.333,Round 7 of 7,0,3.0,10.0,47.619,7,7
1,1951,Juan Manuel Fangio,40,Alfa Romeo,Alfa Romeo,P,4,3,5,5,31.0,86.111,Round 8 of 8,0,6.0,19.355,51.389,8,8
2,1952,Alberto Ascari,34,Ferrari,Ferrari,F P,5,6,6,6,36.0,100.0,Round 6 of 8,2,12.0,33.333,74.306,6,8
3,1953,Alberto Ascari,35,Ferrari,Ferrari,P,6,5,5,4,34.5,95.833,Round 8 of 9,1,6.5,18.841,57.407,8,9
4,1954,Juan Manuel Fangio,43,Maserati,Maserati,P,5,6,7,3,42.0,93.333,Round 7 of 9,2,16.857,40.136,70.547,7,9


## 4.1. Средний, минимальный и максимальный возраст чемпионов
Каждый чемпионский сезон учитывается один раз; возраст дан в полных годах, как в исходной таблице.

In [4]:
display(seasons['Age'].agg(['mean', 'min', 'max']).round(2).to_frame('Возраст, лет'))

,"Возраст, лет"
mean,31.49
min,23.00
max,46.00


## 4.2. Конструктор шасси с наибольшим числом личных титулов
При выступлении чемпиона за два конструктора в одном сезоне учитываем этот сезон у каждого из них; личный титул гонщика при этом не удваивается.

In [5]:
constructor_titles = data.drop_duplicates(['Season', 'Chassis']).groupby('Chassis')['Season'].nunique().sort_values(ascending=False)
display(constructor_titles.to_frame('Личных титулов'))
print('Лидер:', ', '.join(constructor_titles.index[constructor_titles.eq(constructor_titles.max())]), '-', constructor_titles.max(), 'титулов.')

,Личных титулов
Chassis,
Ferrari,15
McLaren,13
Mercedes,9
Red Bull,8
Williams,7
Lotus,6
Brabham,4
Alfa Romeo,2
Renault,2


Лидер: Ferrari - 15 титулов.


## 4.3. Гонщики с наибольшим числом чемпионств

In [6]:
driver_titles = seasons['Driver'].value_counts()
display(driver_titles.to_frame('Титулов'))
print('Рекордсмены:', ', '.join(driver_titles.index[driver_titles.eq(driver_titles.max())]))
print('Количество титулов у каждого:', driver_titles.max())

,Титулов
Driver,
Michael Schumacher,7
Lewis Hamilton,7
Juan Manuel Fangio,5
Alain Prost,4
Sebastian Vettel,4
Max Verstappen,4
Jack Brabham,3
Jackie Stewart,3
Niki Lauda,3


Рекордсмены: Michael Schumacher, Lewis Hamilton
Количество титулов у каждого: 7


## 4.4. Чемпионы, выигравшие менее 30% гонок сезона
Делим число побед на полное число гонок сезона, а не на номер гонки, в которой был обеспечен титул. Условие строгое: меньше 30%.

In [7]:
seasons['Победы, %'] = seasons['Wins'].div(seasons['Races']).mul(100)
under30 = seasons.loc[seasons['Победы, %'].lt(30), ['Season', 'Driver', 'Wins', 'Races', 'Победы, %']]
display(under30.round(2))
print('Гонщики:', ', '.join(under30['Driver'].drop_duplicates()))

,Season,Driver,Wins,Races,"Победы, %"
9,1958,Mike Hawthorn,1,11,9.09
10,1959,Jack Brabham,2,9,22.22
12,1961,Phil Hill,2,8,25.0
15,1964,John Surtees,2,10,20.0
18,1967,Denny Hulme,2,11,18.18
19,1968,Graham Hill,3,12,25.0
25,1974,Emerson Fittipaldi,3,15,20.0
28,1977,Niki Lauda,3,17,17.65
30,1979,Jody Scheckter,3,15,20.0
32,1981,Nelson Piquet,3,15,20.0


Гонщики: Mike Hawthorn, Jack Brabham, Phil Hill, John Surtees, Denny Hulme, Graham Hill, Emerson Fittipaldi, Niki Lauda, Jody Scheckter, Nelson Piquet, Keke Rosberg, Alain Prost, Lewis Hamilton, Sebastian Vettel, Lando Norris


## 4.5. Максимальный промежуток между последовательными титулами
Сортируем чемпионства каждого гонщика по году, затем `diff()` вычисляет разности соседних лет. У однократных чемпионов разность отсутствует; они исключаются.

In [8]:
ordered = seasons[['Driver', 'Season']].sort_values(['Driver', 'Season']).copy()
ordered['Предыдущий титул'] = ordered.groupby('Driver')['Season'].shift()
ordered['Перерыв, лет'] = ordered.groupby('Driver')['Season'].diff()
gaps = ordered.dropna().copy()
gaps[['Предыдущий титул', 'Перерыв, лет']] = gaps[['Предыдущий титул', 'Перерыв, лет']].astype(int)
max_gap = gaps['Перерыв, лет'].max()
result = gaps.loc[gaps['Перерыв, лет'].eq(max_gap)]
display(result)
print(f'Максимальный промежуток между последовательными титулами: {max_gap} лет.')
for _, row in result.iterrows():
    print(f"{row['Driver']}: {row['Предыдущий титул']} → {row['Season']}.")

,Driver,Season,Предыдущий титул,"Перерыв, лет"
35,Niki Lauda,1984,1977,7


Максимальный промежуток между последовательными титулами: 7 лет.
Niki Lauda: 1977 → 1984.
